# 00 — Grand Tour: from `|0>` to surface code

The Phase 9 capstone notebook. Walks through the headline result of every previous phase in 1–2 cells each: state-vector simulator, noise channels, stabilizer simulator, repetition code threshold curve, distance-3 codes, KL conditions, surface-code lattice.

If everything below executes top-to-bottom without error, the whole project is in working order.

Read alongside `notes/12-summary.md`.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qec import statevec as sv
from qec import channels as ch
from qec.pauli import Pauli
from qec.stabilizer import StabilizerState
from qec.codes import repetition as rep
from qec.codes import shor, steane
from qec.codes.surface import SurfaceCode

ATOL = 1e-10
plt.rcParams["figure.figsize"] = (5, 3.5)


## Phase 1 — state-vector simulator: Bell state

In [ ]:
psi = sv.zero_state(2)
psi = sv.apply_1q(psi, sv.H, 0)
psi = sv.cnot(psi, 0, 1)
print("Bell amplitudes:", np.round(psi, 6))


## Phase 2 — depolarizing channel: Bloch ball shrinks uniformly

In [ ]:
plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
rho_in = ch.density_from_state(plus)
ps = np.linspace(0, 1, 21)
fids = [ch.fidelity(ch.apply_channel_full(rho_in, ch.depolarizing(p)), rho_in) for p in ps]
plt.plot(ps, fids, "o-", label="numerical")
plt.plot(ps, 1 - ps/2, "k--", label="analytic 1 - p/2")
plt.xlabel("p"); plt.ylabel("fidelity"); plt.title("depolarizing channel on |+>")
plt.legend(); plt.grid(alpha=0.3); plt.show()


## Phase 3 — stabilizer simulator: GHZ state generators

In [ ]:
st = StabilizerState(4)
st.h(0)
for q in range(1, 4):
    st.cnot(0, q)
print("GHZ stabilizers:")
for g in st.stabilizer_generators():
    print(" ", g)


## Phase 4 — repetition code: P_L = 3 p^2 - 2 p^3

In [ ]:
rng = np.random.default_rng(0)
ps = np.linspace(0, 0.5, 21)
shots = 30_000
mc = [rep.monte_carlo_logical_error(p, shots, rng=rng) for p in ps]
analytic = [rep.analytic_logical_error_rate(p) for p in ps]
plt.plot(ps, ps, "k:", label="no encoding")
plt.plot(ps, analytic, "C0-", label="analytic 3p^2 - 2p^3")
plt.plot(ps, mc, "C1.", label=f"Monte Carlo ({shots})")
plt.xlabel("p"); plt.ylabel("P_L"); plt.title("3-qubit bit-flip code")
plt.legend(); plt.grid(alpha=0.3); plt.show()


## Phase 5 — Steane and Shor: distance 3 against arbitrary Pauli

In [ ]:
print("Steane: weight-1 errors corrected?", steane.all_weight1_errors_corrected())
print("Shor:   weight-1 errors corrected?", shor.all_weight1_errors_corrected())

rng = np.random.default_rng(2026)
ps = np.array([0.005, 0.01, 0.02, 0.03, 0.05, 0.07])
shots = 20_000
steane_pl = [steane.monte_carlo_logical_error(p, shots, rng=rng) for p in ps]
shor_pl = [shor.monte_carlo_logical_error(p, shots, rng=rng) for p in ps]

plt.loglog(ps, ps, "k:", label="no encoding")
plt.loglog(ps, steane_pl, "C2o-", label="Steane [[7,1,3]]")
plt.loglog(ps, shor_pl, "C0s-", label="Shor [[9,1,3]]")
plt.xlabel("p"); plt.ylabel("P_L"); plt.title("distance-3 codes")
plt.legend(); plt.grid(which="both", alpha=0.3); plt.show()


## Phase 6 — CSS condition: H · H^T ≡ 0 (mod 2)

In [ ]:
H = steane.HAMMING_H
print("HAMMING_H:")
print(H)
print()
print("HAMMING_H · HAMMING_H^T (mod 2):")
print((H @ H.T) % 2)


## Phase 8 — surface code: d=3 and d=5

In [ ]:
for d in (3, 5):
    code = SurfaceCode(d=d)
    xs, zs = code.stabilizer_generators()
    LX, LZ = code.logical_x(), code.logical_z()
    print(f"d = {d}: {code.n} data, {len(xs)+len(zs)} stab "
          f"(expected {d*d - 1}), logical weight {LX.weight()} = {LZ.weight()} = d")


## Summary

If you got here without errors, every phase's headline experiment runs end-to-end. The repo is in working order for any future you to come back to.

For deeper notes, see `notes/`. For the full project plan and follow-on directions, see the original plan file or `README.md`.
